<a href="https://colab.research.google.com/github/maha-naveed77/Flyrank-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maha-naveed77/Flyrank-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
#The paper's Random Forest model finds Average Position (43%) and Impressions (32%) as the top predictors of Health Score but the paper itself discloses that Health Score is a composite built directly from Impressions (30 pts), Position (30 pts), CTR (20 pts), and Scroll Depth (20 pts). My methodology question: if the label is partly a linear function of the same features being called "top predictors," how much of that 43%/32% split is the model genuinely discovering a real-world relationship, versus just re-deriving the label's own point-weighting formula? The paper is careful to flag this ("importance is descriptive rather than causal"), which I respect but I'd ask, constructively: would reporting feature importance excluding the components that directly compose the label (i.e., only on inputs like word count, content age, AI sessions) give a cleaner read on what actually drives health independent of its own definition?


In [ ]:
#The paper reports Content Age, Days Since Update, and Days Visible as the strongest signals separating growing from declining pages, with 71% holdout accuracy. My methodology question: the paper's own Trend Direction definition (Finding page 5) buckets pages into up/down/stable/flat/new based on 30-day-vs-previous-30-day impression change — so where does the growth/decline binary label for this specific model come from: is "stable" and "flat" excluded from this classification task entirely, or folded into one of the two classes? If stable/flat pages are dropped, 71% accuracy is measured on an already easier, more separable subset (removing the ambiguous middle), which would make the number look stronger than it would on the full portfolio. I'd also ask whether the 80/20 split was grouped by brand/client — with 57 brands in the portfolio, a plain random split could let the model see the same brand's pages in both train and test, similar to the leakage risk I found in my own w05/w06 client-holdout comparison (0.900 naive vs. 0.820 grouped).

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
import duckdb, pandas as pd, numpy as np
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs;"); con.execute("LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}');")
BASE = "hf://datasets/FlyRank/internship-warehouse"

# Rebuild features (April) + label (May), same as w05
q_apr = f"""
SELECT client_hash_id, content_hash_id,
       SUM(gsc_impressions) AS impressions_apr,
       SUM(gsc_clicks) AS clicks_apr,
       SUM(gsc_sum_position) AS sum_position_apr,
       SUM(sessions_ai) AS sessions_ai_apr,
       SUM(scroll_events) AS scroll_events_apr
FROM read_parquet('{BASE}/fact_content_daily_performance/*/*.parquet')
WHERE month = '2026-04'
GROUP BY 1,2
HAVING SUM(gsc_impressions) > 0
"""
feat_apr = con.execute(q_apr).df()
feat_apr['avg_position_apr'] = feat_apr['sum_position_apr'] / feat_apr['impressions_apr']
feat_apr['ctr_apr'] = feat_apr['clicks_apr'] / feat_apr['impressions_apr']

q_may = f"""
SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_may
FROM read_parquet('{BASE}/fact_content_daily_performance/*/*.parquet')
WHERE month = '2026-05'
GROUP BY 1,2
"""
label_may = con.execute(q_may).df()

df = feat_apr.merge(label_may, on=['client_hash_id','content_hash_id'], how='left')
df['impressions_may'] = df['impressions_may'].fillna(0)
df['is_declining'] = (df['impressions_may'] < df['impressions_apr'] * 0.8).astype(int)

features = ['impressions_apr', 'clicks_apr', 'avg_position_apr', 'ctr_apr', 'sessions_ai_apr', 'scroll_events_apr']

def precision_at_k(df_, score_col, label_col, k=50):
    top_k = df_.sort_values(score_col, ascending=False).head(k)
    return top_k[label_col].mean()

# Rebuild the honest grouped-by-client split + trained model + p_rf from w05
clients = df['client_hash_id'].unique()
np.random.seed(42)
test_clients = np.random.choice(clients, size=int(len(clients)*0.2), replace=False)
train_df = df[~df['client_hash_id'].isin(test_clients)]
test_df = df[df['client_hash_id'].isin(test_clients)].copy()

X_train, y_train = train_df[features].fillna(0), train_df['is_declining']
X_test, y_test = test_df[features].fillna(0), test_df['is_declining']

rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)
test_df['rf_score'] = rf.predict_proba(X_test)[:, 1]
p_rf = precision_at_k(test_df, 'rf_score', 'is_declining', k=50)

print(f"Rebuilt honest grouped-split Precision@50: {p_rf:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rebuilt honest grouped-split Precision@50: 0.820


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

train_naive, test_naive = train_test_split(df, test_size=0.2, random_state=42)
X_train_n, y_train_n = train_naive[features].fillna(0), train_naive['is_declining']
X_test_n, y_test_n = test_naive[features].fillna(0), test_naive['is_declining']

rf_naive = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf_naive.fit(X_train_n, y_train_n)
test_naive = test_naive.copy()
test_naive['rf_score'] = rf_naive.predict_proba(X_test_n)[:, 1]

p_naive = precision_at_k(test_naive, 'rf_score', 'is_declining', k=50)

print(f"BEFORE (naive random split) Precision@50: {p_naive:.3f}")
print(f"AFTER  (grouped-by-client split, from w05) Precision@50: {p_rf:.3f}")

BEFORE (naive random split) Precision@50: 0.900
AFTER  (grouped-by-client split, from w05) Precision@50: 0.820


In [ ]:
#Under a naive random split (ignoring which client each page belongs to), Precision@50 measured 0.900. Under an honest grouped-by-client split — where no client's pages appear in both train and test — Precision@50 dropped to 0.820. This 8-point gap is the model partly learning client-specific quirks (as flagged in the w04 signal audit, where a handful of clients dominated the raw score) rather than purely general decline signal. The grouped number, 0.820, is the one I trust and report going forward — it reflects how the model would likely perform on a genuinely new client it's never seen, which is the real-world use case for this ranking tool.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# Re-check the final feature set for anything derived from the label or a future window
print("Features used:", features)
print()
print("Label definition: is_declining = (impressions_may < impressions_apr * 0.8)")
print()
leak_check = [f for f in features if 'may' in f.lower() or 'score' in f.lower() or 'priority' in f.lower() or 'health' in f.lower()]
print("Suspicious feature names found:", leak_check if leak_check else "None — all features are April-only, pre-label-window.")

Features used: ['impressions_apr', 'clicks_apr', 'avg_position_apr', 'ctr_apr', 'sessions_ai_apr', 'scroll_events_apr']

Label definition: is_declining = (impressions_may < impressions_apr * 0.8)

Suspicious feature names found: None — all features are April-only, pre-label-window.


In [ ]:
#All six features (impressions_apr, clicks_apr, avg_position_apr, ctr_apr, sessions_ai_apr, scroll_events_apr) are aggregated only from April, strictly before the May window used to define the label. No feature name references May or any FlyRank product-decision field (health_score, priority_score, etc.). No leakage identified in the final feature set.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# Bold version (too strong): "The model achieves 90% precision at identifying declining pages."
#Rewritten (safe): "Under an honest, client-grouped validation split — the fairer estimate for how this would perform on a client not seen during training — the model's top 50 ranked pages showed an 82% observed rate of subsequent impression decline in this one month's data (April → May 2026). This is a directional, decision-support result for prioritizing review, not a guarantee about any individual page, and the gap between this (0.820) and a naive split's inflated 0.900 is itself evidence for why the honest split matters."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.